In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from covid import (
    TIME_SERIES_DEATHS_URL,
    country_deaths,
    exponential_model,
    load_deaths,
    logistic_end_day,
    logistic_model,
)

%matplotlib inline

In [ ]:
COUNTRY = "Spain"

# Load the global deaths time series (a single file with a stable schema).
df = load_deaths(TIME_SERIES_DEATHS_URL)

# x -> day number since 1 January 2020, y -> cumulative deaths.
x, y = country_deaths(df, COUNTRY)

print(f"{COUNTRY}: {len(x)} days of data, last day = {int(x[-1])}, deaths = {int(y[-1]):,}")

In [ ]:
fit = curve_fit(logistic_model, x, y, p0=[2, 100, 20000])
a, b, c = fit[0]
a, b, c

In [ ]:
# Standard errors of the fitted logistic parameters.
errors = [np.sqrt(fit[1][i][i]) for i in [0, 1, 2]]
print("parameter std. errors:", [round(e, 2) for e in errors])

# Final day of infection: the day the logistic model reaches 99% of its
# asymptote (the model only reaches c exactly at x == +inf).
end_day = int(np.ceil(logistic_end_day(a, b, c)))
end_day

In [ ]:
# The exponential model only describes the initial outbreak phase, so fit it to
# the first few weeks of non-zero deaths rather than the full (saturated) series.
EXPONENTIAL_WINDOW = 30
nonzero = np.where(y > 0)[0]
x_exp = x[nonzero[:EXPONENTIAL_WINDOW]]
y_exp = y[nonzero[:EXPONENTIAL_WINDOW]]

with np.errstate(over="ignore", invalid="ignore"):
    exp_fit = curve_fit(exponential_model, x_exp, y_exp, p0=[1, 0.1, x_exp[0]], maxfev=10000)
exp_fit

In [ ]:
pred_x = list(range(int(min(x)), end_day))

plt.rcParams["figure.figsize"] = [7, 7]
plt.rc("font", size=14)

# Real data
plt.scatter(x, y, label="Real data", color="red")

# Predicted logistic curve
plt.plot(pred_x, [logistic_model(i, *fit[0]) for i in pred_x], label="Logistic model")

# Predicted exponential curve (only over its valid early-phase domain, extended
# slightly to show how quickly it diverges).
exp_x_plot = list(range(int(x_exp[0]), int(x_exp[-1]) + 20))
plt.plot(
    exp_x_plot, [exponential_model(i, *exp_fit[0]) for i in exp_x_plot], label="Exponential model"
)

plt.legend()
plt.xlabel("Days since 1 January 2020")
plt.ylabel("Total number of deaths")
plt.ylim((min(y) * 0.9, c * 1.1))

plt.show()

In [ ]:
# Residuals. Each model is evaluated on the domain it was designed for:
# - the logistic model over the full series (it captures the saturation), and
# - the exponential model over the early phase (it cannot describe saturation).
y_pred_logistic = [logistic_model(i, *fit[0]) for i in x]
mse_logistic = np.mean((np.asarray(y) - y_pred_logistic) ** 2)

y_pred_exp = [exponential_model(i, *exp_fit[0]) for i in x_exp]
mse_exp = np.mean((np.asarray(y_exp) - y_pred_exp) ** 2)

print(f"logistic RMSE (full series): {np.sqrt(mse_logistic):,.0f}")
print(f"exponential RMSE (early phase): {np.sqrt(mse_exp):,.0f}")